# Tool Calling 실무 과제 — 연습 문제 정답 모음
---
본 노트북은 `ToolCalling_실무용_과제.ipynb`에 수록된 각 시나리오별 연습 문제의 완성형 정답 코드입니다.

## 시나리오 1. 연습 문제 정답

### [문제 1] 주문 취소 도구 `cancel_order(order_id, reason)` 추가
- 조건: `status`가 `"결제완료"`인 주문만 취소 가능, 그 외에는 사유 반환
- `cs_tools`, `CS_TOOL_REGISTRY` 등록 후 테스트

In [ ]:
# [시나리오 1 - 문제 1]
def cancel_order(order_id: str, reason: str) -> dict:
    """주문을 취소한다."""
    order_id = order_id.strip().upper()
    order = ORDERS.get(order_id)
    if order is None:
        return {"success": False, "message": f"주문번호 {order_id}를 찾을 수 없습니다."}
    if order["status"] != "결제완료":
        return {
            "success": False,
            "message": f"현재 상태가 '{order['status']}'이므로 취소할 수 없습니다. 출고 전(결제완료) 주문만 취소 가능합니다."
        }
    order["status"] = "취소완료"
    order["cancel_reason"] = reason
    return {"success": True, "order_id": order_id, "message": "주문이 성공적으로 취소되었습니다. 환불은 3영업일 내 처리됩니다."}

cancel_tool = {
    "type": "function",
    "function": {
        "name": "cancel_order",
        "description": "주문을 취소한다. 출고 전(결제완료) 상태의 주문만 취소 가능하다. 고객이 주문 취소를 명확히 요청한 경우에만 사용한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "주문번호 (예: ORD-1003)"},
                "reason": {"type": "string", "description": "고객이 밝힌 취소 사유. 언급이 없으면 '미기재'"},
            },
            "required": ["order_id", "reason"],
        },
    },
}

if not any(t["function"]["name"] == "cancel_order" for t in cs_tools):
    cs_tools.append(cancel_tool)
CS_TOOL_REGISTRY["cancel_order"] = cancel_order

print("=== [시나리오 1 - 문제 1 테스트] ===")
for q in ["ORD-1003 취소해 주세요. 단순변심이에요.", "ORD-1001 취소할래요."]:
    print("고객:", q)
    print("챗봇:", ask_cs_bot(q), "\n")

### [문제 2] `get_return_policy`의 description을 짧게 바꾼 후 동작 비교
- description이 모호해지면 모델이 언제 해당 도구를 써야 하는지 판단하기 어려워져, 연쇄 도구 호출(주문 조회 후 반품 정책 확인)이 누락되거나 불완전한 답변을 낼 가능성이 높아집니다.

In [ ]:
# [시나리오 1 - 문제 2]
# description을 '정책 조회'로 임시 변경
for t in cs_tools:
    if t["function"]["name"] == "get_return_policy":
        orig_desc = t["function"]["description"]
        t["function"]["description"] = "정책 조회"

print("=== [시나리오 1 - 문제 2 테스트: 모호한 description] ===")
test_q = "ORD-1002로 셔츠 샀는데 사이즈가 안 맞아요. 반품 가능한가요?"
print("고객:", test_q)
print("챗봇:", ask_cs_bot(test_q))

# 원복
for t in cs_tools:
    if t["function"]["name"] == "get_return_policy":
        t["function"]["description"] = orig_desc

## 시나리오 2. 연습 문제 정답

### [문제 1] 예약 취소 도구 `cancel_reservation(reservation_id)` 추가 (쓰기 작업, 승인 포함)

In [ ]:
# [시나리오 2 - 문제 1]
def cancel_reservation(reservation_id: str) -> dict:
    """예약을 취소한다."""
    reservation_id = reservation_id.strip().upper()
    for i, r in enumerate(RESERVATIONS):
        if r["id"] == reservation_id:
            removed = RESERVATIONS.pop(i)
            return {
                "success": True,
                "message": f"예약 {reservation_id}('{removed['title']}')이 취소되었습니다.",
                "cancelled_reservation": removed,
            }
    raise ValueError(f"예약 ID '{reservation_id}'를 찾을 수 없습니다.")

cancel_res_tool = {
    "type": "function",
    "function": {
        "name": "cancel_reservation",
        "description": "회의실 예약을 취소한다. 되돌릴 수 없는 작업이므로 예약 번호를 명확히 확인한 후 호출한다.",
        "strict": True,
        "parameters": {
            "type": "object",
            "properties": {
                "reservation_id": {"type": "string", "description": "취소할 예약 번호 (예: RSV-001)"}
            },
            "required": ["reservation_id"],
            "additionalProperties": False,
        },
    },
}

if not any(t["function"]["name"] == "cancel_reservation" for t in room_tools):
    room_tools.append(cancel_res_tool)
ROOM_TOOL_REGISTRY["cancel_reservation"] = cancel_reservation
WRITE_TOOLS.add("cancel_reservation")

print("=== [시나리오 2 - 문제 1 테스트] ===")
history = new_room_session()
q = "RSV-001 예약 취소해줘"
print("직원:", q)
print("비서:", room_chat(history, q))

### [문제 2] `check_room_availability`에 `required_equipment` 파라미터 추가
- Strict 모드 준수 (`additionalProperties: false`, required 목록 포함, nullable 표현)

In [ ]:
# [시나리오 2 - 문제 2]
# 1) 함수 재정의
def check_room_availability(date: str, start_time: str, end_time: str, min_capacity=None, required_equipment=None) -> dict:
    """특정 날짜·시간대에 예약 가능한 회의실 목록을 반환한다."""
    _validate_date(date)
    _validate_time_range(start_time, end_time)

    available = []
    for room_id, room in ROOMS.items():
        if min_capacity and room["capacity"] < min_capacity:
            continue
        if required_equipment and required_equipment not in room.get("equipment", []):
            continue
        busy = any(
            r["room_id"] == room_id and r["date"] == date
            and _is_overlap(start_time, end_time, r["start_time"], r["end_time"])
            for r in RESERVATIONS
        )
        if not busy:
            available.append({"room_id": room_id, **room})

    return {
        "date": date,
        "time": f"{start_time}~{end_time}",
        "available_count": len(available),
        "available_rooms": available,
    }

ROOM_TOOL_REGISTRY["check_room_availability"] = check_room_availability

# 2) 스키마 업데이트 (Strict 모드 규칙: required_equipment를 속성 정의 및 required에 추가)
for t in room_tools:
    if t["function"]["name"] == "check_room_availability":
        t["function"]["parameters"]["properties"]["required_equipment"] = {
            "type": ["string", "null"],
            "description": "필요한 회의실 장비 (예: '화상회의', '빔프로젝터', '마이크', '화이트보드', 'TV'). 언급 없으면 null",
        }
        if "required_equipment" not in t["function"]["parameters"]["required"]:
            t["function"]["parameters"]["required"].append("required_equipment")
        break

print("=== [시나리오 2 - 문제 2 테스트] ===")
history = new_room_session()
q = "내일 오전 10시에 화상회의 되는 방 있어?"
print("직원:", q)
print("비서:", room_chat(history, q))

### [문제 3] `parallel_tool_calls=False`로 실행 후 API 호출 횟수 비교
- 병렬 호출이 꺼지면 독립적인 조회라도 모델이 1건씩 순차적으로 요청하므로 전체 대화 라운드 수가 증가합니다.

In [ ]:
# [시나리오 2 - 문제 3]
def room_chat_sequential(history: list, user_text: str, approval_fn=auto_approve, max_rounds: int = 10) -> str:
    history.append({"role": "user", "content": user_text})
    call_rounds = 0

    for _ in range(max_rounds):
        call_rounds += 1
        r = client.chat.completions.create(
            model=MODEL,
            messages=history,
            tools=room_tools,
            parallel_tool_calls=False,   # 병렬 호출 비활성화
        )
        msg = r.choices[0].message

        if not msg.tool_calls:
            history.append({"role": "assistant", "content": msg.content})
            print(f"  → 총 API 호출(라운드) 수: {call_rounds}")
            return msg.content

        history.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })
        print(f"  [라운드 {call_rounds}] 도구 호출 수: {len(msg.tool_calls)}건 -> {msg.tool_calls[0].function.name}")
        for tc in msg.tool_calls:
            result = execute_tool_call(tc, ROOM_TOOL_REGISTRY, approval_fn, WRITE_TOOLS)
            history.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result, ensure_ascii=False),
            })

    return "요청 처리 단계 초과"

print("=== [시나리오 2 - 문제 3 테스트: parallel_tool_calls=False] ===")
history = new_room_session()
q = "내일, 다음주 월요일, 다음주 화요일 각각 오후 2시~3시에 8명 이상 들어가는 회의실 있는지 한 번에 알려줘"
print("직원:", q)
print("비서:", room_chat(room_chat_sequential(history, q) if False else history, q))

## 시나리오 3. 연습 문제 정답

### [문제 1] `send_email` 도구에 사람 승인(Human-in-the-loop) 적용

In [ ]:
# [시나리오 3 - 문제 1]
def run_agent_with_approval(
    user_input: str,
    system_prompt: str = AGENT_SYSTEM_PROMPT,
    tools: list = agent_tools,
    registry: dict = AGENT_TOOL_REGISTRY,
    needs_approval: set = frozenset(["send_email"]),
    approval_fn = auto_approve,
    max_iterations: int = 10,
    verbose: bool = True,
) -> dict:
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_input}]
    trace = []
    usage = {"prompt_tokens": 0, "completion_tokens": 0}
    started = time.time()

    for step in range(1, max_iterations + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        msg = response.choices[0].message
        usage["prompt_tokens"] += response.usage.prompt_tokens
        usage["completion_tokens"] += response.usage.completion_tokens

        if not msg.tool_calls:
            if verbose:
                print(f"[Step {step}] 최종 답변 완료")
            return {"answer": msg.content, "steps": step, "trace": trace,
                    "usage": usage, "elapsed_sec": round(time.time() - started, 1)}

        messages.append({"role": "assistant", "content": msg.content,
                         "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})

        if verbose:
            print(f"[Step {step}] 도구 호출 {len(msg.tool_calls)}건")
        for tc in msg.tool_calls:
            result = execute_tool_call(tc, registry, approval_fn=approval_fn, needs_approval=needs_approval)
            is_error = isinstance(result, dict) and ("error" in result or result.get("status") == "rejected")
            trace.append({"step": step, "tool": tc.function.name,
                          "args": json.loads(tc.function.arguments), "error": is_error, "result": result})
            if verbose:
                flag = "  [ERROR/REJECTED]" if is_error else ""
                print(f"    - {tc.function.name}({tc.function.arguments}){flag}")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(result, ensure_ascii=False)})

    return {"answer": "중단", "steps": max_iterations, "trace": trace, "usage": usage,
            "elapsed_sec": round(time.time() - started, 1)}

OUTBOX.clear()
print("=== [시나리오 3 - 문제 1 테스트: 메일 발송 승인 절차] ===")
res = run_agent_with_approval(
    "9월 서울 매출 확인해서 영업팀장님께 간단히 메일 보내줘.",
    approval_fn=auto_approve,
)
print("답변:", res["answer"])

### [문제 2] 3개월 추세 분석 도구 `get_sales_trend` 추가 및 꾸준한 성장 지역 선별

In [ ]:
# [시나리오 3 - 문제 2]
def get_sales_trend(region: str, months: list) -> dict:
    """지정된 지역의 월별 매출 추세 및 연속 성장 여부를 분석한다."""
    history = []
    for m in months:
        if m not in SALES_DB:
            raise ValueError(f"{m} 데이터가 DB에 없습니다.")
        if region not in SALES_DB[m]:
            raise ValueError(f"'{region}' 지역이 {m} 데이터에 없습니다.")
        history.append({"month": m, "sales": SALES_DB[m][region]})

    consistently_growing = all(
        history[i]["sales"] < history[i + 1]["sales"]
        for i in range(len(history) - 1)
    )

    return {
        "region": region,
        "trend": history,
        "is_consistently_growing": consistently_growing,
    }

trend_tool = {
    "type": "function",
    "function": {
        "name": "get_sales_trend",
        "description": "지정된 지역과 월 리스트에 대해 매출 추세를 조회하고 지속적인 성장 여부를 확인한다.",
        "strict": True,
        "parameters": {
            "type": "object",
            "properties": {
                "region": {"type": "string", "description": "조회할 지역명 (서울, 부산, 대구, 광주, 대전)"},
                "months": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "조회할 월 목록 (예: ['2026-07', '2026-08', '2026-09'])",
                },
            },
            "required": ["region", "months"],
            "additionalProperties": False,
        },
    },
}

extended_agent_tools = agent_tools + [trend_tool]
extended_registry = {**AGENT_TOOL_REGISTRY, "get_sales_trend": get_sales_trend}

print("=== [시나리오 3 - 문제 2 테스트] ===")
res = run_agent(
    "7월, 8월, 9월 3개월 동안 꾸준히 성장한 지역만 골라줘.",
    tools=extended_agent_tools,
    registry=extended_registry,
)
print("답변:\n", res["answer"])

### [문제 3] 병렬 원칙을 제거했을 때 단계 수 및 토큰 사용량 비교

In [ ]:
# [시나리오 3 - 문제 3]
PROMPT_NO_PARALLEL = """당신은 캠퍼스몰 영업기획팀의 데이터 분석 에이전트입니다.
[작업 원칙]
1. 숫자는 반드시 get_sales_data로 조회한 값만 사용하고, 성장률 등 계산은 calculate_growth_rate 도구로 수행하세요.
2. 메일 수신자는 find_employee로 확인한 주소만 사용하세요.
3. 도구가 error를 반환하면 원인을 파악해 수정하거나, 불가능하면 사용자에게 이유를 설명하세요.
4. 작업이 끝나면 수행한 내용을 3~5줄로 요약 보고하세요."""

OUTBOX.clear()
task = "8월 대비 9월 지역별 매출 성장률을 계산하고 가장 높은/낮은 지역을 팀장님께 보고 메일 보내줘."

print("=== [시나리오 3 - 문제 3 테스트: 병렬 규칙 제거] ===")
res_no_parallel = run_agent(task, system_prompt=PROMPT_NO_PARALLEL, verbose=False)
print(f"[병렬 원칙 제외] 단계 수: {res_no_parallel['steps']}스텝 | 토큰 사용량: {res_no_parallel['usage']} | 소요 시간: {res_no_parallel['elapsed_sec']}초")